# CTC OCR Training for Upper Plates
- สคริปต์นี้เตรียม train/val จาก `upper_train`, และประเมินบน `upper_test` (real) กับ `plate_upper_synth_test` (synthetic).
- สถาปัตยกรรม: CRNN (CNN + BiLSTM) + CTC loss พร้อม greedy decode สำหรับ metric เบื้องต้น.
- ปรับ hyperparameters (batch size, epochs, LR) ให้เหมาะกับ RTX 3080 ตาม resource ที่มี.

**Colab mode:**
- Mount Google Drive, unzip `upper_train.zip`, `upper_test.zip`, `plate_upper_synth_test.zip`
- Save best checkpoint to Drive (new folder)

In [24]:
# Cell 1: Dataset paths (Local laptop: use existing folders; Colab: optional zip/unzip)
import os, zipfile, shutil
from pathlib import Path

def is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = is_colab()
print('IS_COLAB:', IS_COLAB)

def find_repo_root(start: Path) -> Path:
    """Walk upwards to find a folder containing data/upper_train/labels.csv."""
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'data' / 'upper_train' / 'labels.csv').exists():
            return p
    return start

def resolve_local_data_root() -> str:
    repo_root = find_repo_root(Path.cwd())
    return str((repo_root / 'data').resolve())

def unzip_to(zip_path: str, out_dir: str):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f'Zip not found: {zip_path}')
    if os.path.exists(out_dir) and os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f'Skip (already exists): {out_dir}')
        return
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print(f'Unzipped: {zip_path} -> {out_dir}')

def resolve_dataset_dir(root_dir: str) -> str:
    # Supports both layouts:
    # 1) root_dir/labels.csv + root_dir/data/...
    # 2) root_dir/<subdir>/labels.csv + ...
    if os.path.exists(os.path.join(root_dir, 'labels.csv')):
        return root_dir
    for d in os.listdir(root_dir):
        sd = os.path.join(root_dir, d)
        if os.path.isdir(sd) and os.path.exists(os.path.join(sd, 'labels.csv')):
            return sd
    raise FileNotFoundError(f'Could not find labels.csv under: {root_dir}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_TRAIN = '/content/drive/MyDrive/ALPRV2/upper_train.zip'
    ZIP_TEST = '/content/drive/MyDrive/ALPRV2/upper_test.zip'
    ZIP_TEST_SYN = '/content/drive/MyDrive/ALPRV2/plate_upper_synth_test.zip'

    OUT_ROOT = '/content/datasets'
    RAW_TRAIN_DIR = os.path.join(OUT_ROOT, 'upper_train')
    RAW_TEST_DIR = os.path.join(OUT_ROOT, 'upper_test')
    RAW_TEST_SYN_DIR = os.path.join(OUT_ROOT, 'plate_upper_synth_test')
    os.makedirs(OUT_ROOT, exist_ok=True)

    unzip_to(ZIP_TRAIN, RAW_TRAIN_DIR)
    unzip_to(ZIP_TEST, RAW_TEST_DIR)
    unzip_to(ZIP_TEST_SYN, RAW_TEST_SYN_DIR)

    TRAIN_DIR = resolve_dataset_dir(RAW_TRAIN_DIR)
    TEST_DIR = resolve_dataset_dir(RAW_TEST_DIR)
    TEST_SYN_DIR = resolve_dataset_dir(RAW_TEST_SYN_DIR)

    DATA_ROOT_PATH = '/content/datasets'
    SAVE_DIR = '/content/drive/MyDrive/ALPR_Project/alpr_ctc_upper'
else:
    DATA_ROOT_PATH = resolve_local_data_root()
    SAVE_DIR = str((Path.cwd() / 'weights').resolve())

print('DATA_ROOT_PATH:', DATA_ROOT_PATH)
print('SAVE_DIR      :', SAVE_DIR)

IS_COLAB: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Skip (already exists): /content/datasets/upper_train
Skip (already exists): /content/datasets/upper_test
Skip (already exists): /content/datasets/plate_upper_synth_test
DATA_ROOT_PATH: /content/datasets
SAVE_DIR      : /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper


In [25]:
from __future__ import annotations

from pathlib import Path
import math
import random
import json
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path(DATA_ROOT_PATH)
TRAIN_LABELS = DATA_ROOT / "upper_train" / "upper_train" / "labels.csv"
TRAIN_IMAGES = DATA_ROOT / "upper_train" / "upper_train" / "data"
TEST_REAL_LABELS = DATA_ROOT / "upper_test" / "upper_test" / "labels.csv"
TEST_REAL_IMAGES = DATA_ROOT / "upper_test" / "upper_test" / "data"
TEST_SYN_LABELS = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "labels.csv"
TEST_SYN_IMAGES = DATA_ROOT / "plate_upper_synth_test" / "plate_upper_synth_test" / "data"

IMG_HEIGHT = 32
IMG_WIDTH = 128
BATCH_SIZE = 128
EPOCHS = 20
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 8
TRAIN_FRAC = 0.9
GRAD_CLIP = 5.0

if 'IS_COLAB' in globals() and IS_COLAB:
    BATCH_SIZE = 128
    NUM_WORKERS = 2
    print("Colab overrides: BATCH_SIZE=128, NUM_WORKERS=2")

CHECKPOINT_PATH = Path(SAVE_DIR) / "upper_ctc_best.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Colab overrides: BATCH_SIZE=128, NUM_WORKERS=2
Device: cuda


In [26]:
def load_labels(csv_path: Path, images_dir: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "label" not in df.columns or "filename" not in df.columns:
        raise ValueError("labels.csv must contain 'filename' and 'label' columns")
    df["filename"] = df["filename"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    df["full_path"] = df["filename"].apply(lambda x: images_dir / x)
    missing = (~df["full_path"].apply(Path.exists)).sum()
    if missing:
        print(f"Warning: {missing} files missing for {csv_path}")
    return df

df_full = load_labels(TRAIN_LABELS, TRAIN_IMAGES)
df_full = df_full.sample(frac=1, random_state=SEED).reset_index(drop=True)
cut = int(len(df_full) * TRAIN_FRAC)
df_train = df_full.iloc[:cut].copy()
df_val = df_full.iloc[cut:].copy()
print("Train/Val split:", len(df_train), len(df_val))

df_test_real = load_labels(TEST_REAL_LABELS, TEST_REAL_IMAGES)
df_test_synth = load_labels(TEST_SYN_LABELS, TEST_SYN_IMAGES)

/tmp/ipython-input-2317082158.py:2: DtypeWarning: Columns (3,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Train/Val split: 142349 15817


In [27]:
ALLOWED_CHARS = list("0123456789กขคฆงจฉชซฌญฎฏฐฑฒณดตถทธนบปผฝพฟภมยรลวศษสหฬอฮ")  # exclude ฃ ฅ

def filter_unsupported(df: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    def has_unsupported(s: str) -> bool:
        return any(ch not in ALLOWED_CHARS for ch in str(s))
    mask = df[label_col].apply(has_unsupported)
    if mask.any():
        print(f"Filtered {mask.sum()} rows with unsupported chars in {label_col}")
    return df[~mask].copy()

df_train = filter_unsupported(df_train)
df_val = filter_unsupported(df_val)
df_test_real = filter_unsupported(df_test_real)
df_test_synth = filter_unsupported(df_test_synth)

def build_charset_fixed(labels: list[str]) -> tuple[list[str], dict[str, int]]:
    unknown = set()
    for text in labels:
        for ch in str(text):
            if ch not in ALLOWED_CHARS:
                unknown.add(ch)
    if unknown:
        preview = " ".join(sorted(list(unknown))[:30])
        print(f"Warning: found unsupported chars (ignored in charset): {preview}")
    idx_to_char = ["<BLANK>"] + ALLOWED_CHARS
    char_to_idx = {c: i for i, c in enumerate(idx_to_char)}
    return idx_to_char, char_to_idx

all_labels = pd.concat([
    df_train["label"],
    df_val["label"],
    df_test_real["label"],
    df_test_synth["label"],
], ignore_index=True).tolist()

idx_to_char, char_to_idx = build_charset_fixed(all_labels)
print("Charset size (incl. blank):", len(idx_to_char))
print("Charset chars:", " ".join(idx_to_char[1:]))

Filtered 20 rows with unsupported chars in label
Filtered 1 rows with unsupported chars in label
Filtered 2 rows with unsupported chars in label
Charset size (incl. blank): 53
Charset chars: 0 1 2 3 4 5 6 7 8 9 ก ข ค ฆ ง จ ฉ ช ซ ฌ ญ ฎ ฏ ฐ ฑ ฒ ณ ด ต ถ ท ธ น บ ป ผ ฝ พ ฟ ภ ม ย ร ล ว ศ ษ ส ห ฬ อ ฮ


In [28]:
class OCRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["full_path"]).convert("L")
        # Assert size already 128x32 to avoid unintended resizing
        assert img.size == (IMG_WIDTH, IMG_HEIGHT), f"Unexpected size {img.size} for {row['full_path']}"
        img = self.transform(img)
        label_text = row["label"]
        target = torch.tensor([char_to_idx[c] for c in label_text], dtype=torch.long)
        return img, target, label_text

train_transform = T.Compose([
    T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02)], p=0.7),
    T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=0.2),
    T.RandomAffine(degrees=2, translate=(0.02, 0.05), scale=(0.95, 1.05), shear=1, fill=0),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

def collate_fn(batch):
    imgs, targets, texts = zip(*batch)
    imgs = torch.stack(imgs)
    target_lengths = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    targets = torch.cat(targets)
    return imgs, targets, target_lengths, list(texts)

train_ds = OCRDataset(df_train, train_transform)
val_ds = OCRDataset(df_val, eval_transform)
test_real_ds = OCRDataset(df_test_real, eval_transform)
test_synth_ds = OCRDataset(df_test_synth, eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
    prefetch_factor=4,           # เพิ่ม: โหลดล่วงหน้า 4 batch/worker
    persistent_workers=True      # เพิ่ม: ไม่ restart workers ทุก epoch
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)
test_real_loader = DataLoader(test_real_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)
test_synth_loader = DataLoader(test_synth_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn, prefetch_factor=4, persistent_workers=True)


In [29]:
class CRNN(nn.Module):
    def __init__(self, num_classes: int, hidden: int = 256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.rnn = nn.LSTM(512 * (IMG_HEIGHT // 16), hidden, num_layers=2, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden * 2, num_classes)

    @staticmethod
    def seq_len_from_width(width: int) -> int:
        # width reduced by factor 4 from two MaxPool2d(2,2) layers
        return width // 4

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.cnn(x)
        b, c, h, w = feats.size()
        feats = feats.permute(0, 3, 1, 2).contiguous()
        feats = feats.view(b, w, c * h)
        rnn_out, _ = self.rnn(feats)
        logits = self.classifier(rnn_out)  # (B, T, C)
        return logits.permute(1, 0, 2)  # (T, B, C) for CTC

In [30]:
def greedy_decode(logits: torch.Tensor) -> list[str]:
    # logits: (T, B, C)
    probs = logits.softmax(2)
    indices = probs.argmax(2).permute(1, 0)  # (B, T)
    texts = []
    for seq in indices:
        prev = None
        chars = []
        for idx in seq.tolist():
            if idx != 0 and idx != prev:
                chars.append(idx_to_char[idx])
            prev = idx
        texts.append("".join(chars))
    return texts

def step(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    model.train(is_train)
    for imgs, targets, target_lengths, texts in loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(imgs)
        log_probs = logits.log_softmax(2)
        input_lengths = torch.full((imgs.size(0),), logits.size(0), dtype=torch.long, device=device)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = greedy_decode(log_probs.detach())
        total_correct += sum(p == t for p, t in zip(preds, texts))
        total_count += len(texts)
    avg_loss = total_loss / max(1, total_count)
    acc = total_correct / max(1, total_count)
    return avg_loss, acc

def evaluate(model, loader, criterion, tag: str):
    with torch.no_grad():
        loss, acc = step(model, loader, criterion, optimizer=None)
    print(f"[{tag}] loss={loss:.4f} acc={acc:.4f}")
    return loss, acc

In [31]:
model = CRNN(num_classes=len(idx_to_char)).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=EPOCHS)

best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    for step_idx, (imgs, targets, target_lengths, texts) in enumerate(tqdm(train_loader, desc="train", leave=False), start=1):
        imgs = imgs.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)
        logits = model(imgs)
        log_probs = logits.log_softmax(2)
        input_lengths = torch.full((imgs.size(0),), logits.size(0), dtype=torch.long, device=device)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * imgs.size(0)
        preds = greedy_decode(log_probs.detach())
        total_correct += sum(p == t for p, t in zip(preds, texts))
        total_count += len(texts)
        if step_idx % 200 == 0:
            running_loss = total_loss / max(1, total_count)
            running_acc = total_correct / max(1, total_count)
            print(f"  step {step_idx}: running loss={running_loss:.4f} acc={running_acc:.4f}")
    train_loss = total_loss / max(1, total_count)
    train_acc = total_correct / max(1, total_count)
    print(f"[train] loss={train_loss:.4f} acc={train_acc:.4f}")

    val_loss, val_acc = evaluate(model, val_loader, criterion, tag="val")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "idx_to_char": idx_to_char,
            "char_to_idx": char_to_idx,
            "config": {
                "IMG_HEIGHT": IMG_HEIGHT,
                "IMG_WIDTH": IMG_WIDTH,
            },
        }, CHECKPOINT_PATH)
        print(f"Saved new best to {CHECKPOINT_PATH}")

Epoch 1/20


train:   0%|          | 0/1112 [00:00<?, ?it/s]

  step 200: running loss=9.5913 acc=0.0000
  step 400: running loss=6.6774 acc=0.0000
  step 600: running loss=5.6125 acc=0.0000
  step 800: running loss=5.0517 acc=0.0000
  step 1000: running loss=4.7059 acc=0.0000
[train] loss=4.5627 acc=0.0000
[val] loss=3.2451 acc=0.0000
Epoch 2/20


train:   0%|          | 0/1112 [00:00<?, ?it/s]

  step 200: running loss=3.1760 acc=0.0000
  step 400: running loss=2.9519 acc=0.0000
  step 600: running loss=2.6066 acc=0.0001
  step 800: running loss=2.3048 acc=0.0017
  step 1000: running loss=2.0767 acc=0.0066
[train] loss=1.9709 acc=0.0106
[val] loss=0.9426 acc=0.0617
Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper/upper_ctc_best.pt
Epoch 3/20


...

  step 200: running loss=0.0061 acc=0.9898
  step 400: running loss=0.0058 acc=0.9901
  step 600: running loss=0.0058 acc=0.9899
  step 800: running loss=0.0058 acc=0.9900
  step 1000: running loss=0.0057 acc=0.9901
[train] loss=0.0057 acc=0.9902
[val] loss=0.0050 acc=0.9933
Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper/upper_ctc_best.pt
Epoch 19/20


train:   0%|          | 0/1112 [00:00<?, ?it/s]

  step 200: running loss=0.0054 acc=0.9909
  step 400: running loss=0.0052 acc=0.9908
  step 600: running loss=0.0051 acc=0.9911
  step 800: running loss=0.0051 acc=0.9909
  step 1000: running loss=0.0050 acc=0.9912
[train] loss=0.0050 acc=0.9913
[val] loss=0.0050 acc=0.9934
Saved new best to /content/drive/MyDrive/ALPR_Project/alpr_ctc_upper/upper_ctc_best.pt
Epoch 20/20


train:   0%|          | 0/1112 [00:00<?, ?it/s]

  step 200: running loss=0.0050 acc=0.9913
  step 400: running loss=0.0047 acc=0.9918
  step 600: running loss=0.0048 acc=0.9915
  step 800: running loss=0.0049 acc=0.9915
  step 1000: running loss=0.0048 acc=0.9916
[train] loss=0.0049 acc=0.9916
[val] loss=0.0049 acc=0.9934


In [32]:
def load_best(model_path: Path, device: torch.device):
    ckpt = torch.load(model_path, map_location=device)
    model = CRNN(num_classes=len(ckpt["idx_to_char"])).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt["idx_to_char"]

best_model, _ = load_best(CHECKPOINT_PATH, device) if CHECKPOINT_PATH.exists() else (model, idx_to_char)

evaluate(best_model, test_real_loader, criterion, tag="test_real")
evaluate(best_model, test_synth_loader, criterion, tag="test_synth")

[test_real] loss=0.0484 acc=0.9657
[test_synth] loss=0.0015 acc=0.9982


(0.001497655747400131, 0.9982)